# House Prices — Modeling

Kaggle House Prices: Advanced Regression Techniques

**Objective:** Build, evaluate, optimize, and ensemble regression models using fixed K-fold cross-validation and RMSLE.

## Phase 1 — Modeling Setup

In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error, r2_score

RANDOM_STATE = 42
N_SPLITS = 5

TRAIN_PATH = "../data/processed/train_engineered.csv"
TEST_PATH = "../data/processed/test_engineered.csv"

TARGET = "SalePrice"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

X = train.drop(columns=[TARGET])
y = np.log1p(train[TARGET])

X_test = test.copy()

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(f"Training data: {X.shape}")
print(f"Test data: {X_test.shape}")
print(f"Target: {TARGET}")
print(f"CV folds: {N_SPLITS}")

Training data: (1460, 94)
Test data: (1459, 94)
Target: SalePrice
CV folds: 5


In [16]:
def evaluate_model(model, X, y, cv):
    oof_predictions = np.zeros(len(X))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X), 1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model.fit(X_train, y_train)

        valid_pred_log = model.predict(X_valid)
        valid_pred = np.maximum(np.expm1(valid_pred_log), 0)
        actual = np.maximum(np.expm1(y_valid), 0)

        score = np.sqrt(mean_squared_log_error(actual, valid_pred))

        oof_predictions[valid_idx] = valid_pred_log
        fold_scores.append(score)

        print(f"Fold {fold}: RMSLE = {score:.5f}")

    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)

    print(f"\nMean RMSLE: {mean_score:.5f}")
    print(f"Std RMSLE:  {std_score:.5f}")
    print(f"R² Score:   {r2_score(actual, valid_pred):.5f}")

    return oof_predictions, fold_scores, mean_score, std_score

## Phase 2 — Baseline Models

In [17]:
ordinal_mappings = {
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual": ["NoFeature", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond": ["NoFeature", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtExposure": ["NoFeature", "No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["NoFeature", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["NoFeature", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "FireplaceQu": ["NoFeature", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageFinish": ["NoFeature", "Unf", "RFn", "Fin"],
    "GarageQual": ["NoFeature", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond": ["NoFeature", "Po", "Fa", "TA", "Gd", "Ex"],
    "PavedDrive": ["N", "P", "Y"],
    "Functional": ["Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "PoolQC": ["NoFeature", "Fa", "Gd", "Ex"],
    "Fence": ["NoFeature", "MnWw", "GdWo", "MnPrv", "GdPrv"]
}

ordinal_cols = list(ordinal_mappings.keys())

categorical_cols = X.select_dtypes(include=["object", "str"]).columns.tolist()

nominal_cols = [
    col for col in categorical_cols
    if col not in ordinal_cols
]

categorical_cols_to_onehot = nominal_cols + [
    "MSSubClass",
    "MoSold",
    "YrSold"
]

numerical_cols = [
    col for col in X.columns
    if col not in ordinal_cols + categorical_cols_to_onehot + ["Id"]
]

print(f"Numerical: {len(numerical_cols)}")
print(f"Ordinal: {len(ordinal_cols)}")
print(f"Nominal + categorical integers: {len(categorical_cols_to_onehot)}")

Numerical: 47
Ordinal: 17
Nominal + categorical integers: 29


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

ordinal_categories = [
    ordinal_mappings[col] for col in ordinal_cols
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_cols
        ),
        (
            "ord",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            ordinal_cols
        ),
        (
            "nom",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols_to_onehot
        )
    ],
    remainder="drop"
)

ridge = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10.0))
])

ridge_oof, ridge_fold_scores, ridge_mean, ridge_std = evaluate_model(
    ridge,
    X,
    y,
    kf
)

Fold 1: RMSLE = 0.13254
Fold 2: RMSLE = 0.12754
Fold 3: RMSLE = 0.22162
Fold 4: RMSLE = 0.12231
Fold 5: RMSLE = 0.10908

Mean RMSLE: 0.14262
Std RMSLE:  0.04027
R² Score:   0.92903


In [19]:
results = pd.DataFrame([
    {
        "Model": "Ridge",
        "Mean RMSLE": ridge_mean,
        "Std RMSLE": ridge_std
    }
])

results

,Model,Mean RMSLE,Std RMSLE
0,Ridge,0.142617,0.040268


In [20]:
from sklearn.ensemble import ExtraTreesRegressor

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "ord",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            ordinal_cols
        ),
        (
            "nom",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols_to_onehot
        )
    ],
    remainder="passthrough"
)

extra_trees = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        ExtraTreesRegressor(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

extra_trees_oof, extra_trees_fold_scores, extra_trees_mean, extra_trees_std = evaluate_model(
    extra_trees,
    X,
    y,
    kf
)

Fold 1: RMSLE = 0.13774
Fold 2: RMSLE = 0.12835
Fold 3: RMSLE = 0.16839
Fold 4: RMSLE = 0.13521
Fold 5: RMSLE = 0.11468

Mean RMSLE: 0.13688
Std RMSLE:  0.01768
R² Score:   0.91138


In [21]:
results.loc[len(results)] = [
    "Extra Trees",
    extra_trees_mean,
    extra_trees_std
]

results.sort_values("Mean RMSLE").reset_index(drop=True)

,Model,Mean RMSLE,Std RMSLE
0,Extra Trees,0.136876,0.017676
1,Ridge,0.142617,0.040268


In [22]:
from sklearn.ensemble import GradientBoostingRegressor

gradient_boosting = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        GradientBoostingRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=3,
            random_state=RANDOM_STATE
        )
    )
])

gb_oof, gb_fold_scores, gb_mean, gb_std = evaluate_model(
    gradient_boosting,
    X,
    y,
    kf
)

Fold 1: RMSLE = 0.13420
Fold 2: RMSLE = 0.11842
Fold 3: RMSLE = 0.16217
Fold 4: RMSLE = 0.12577
Fold 5: RMSLE = 0.10890

Mean RMSLE: 0.12989
Std RMSLE:  0.01816
R² Score:   0.92079


In [23]:
results.loc[len(results)] = [
    "Gradient Boosting",
    gb_mean,
    gb_std
]

results = results.sort_values("Mean RMSLE").reset_index(drop=True)
results

,Model,Mean RMSLE,Std RMSLE
0,Gradient Boosting,0.129890,0.018164
1,Extra Trees,0.136876,0.017676
2,Ridge,0.142617,0.040268


In [24]:
from xgboost import XGBRegressor

In [25]:
xgb = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

xgb_oof, xgb_fold_scores, xgb_mean, xgb_std = evaluate_model(
    xgb,
    X,
    y,
    kf
)

Fold 1: RMSLE = 0.12533
Fold 2: RMSLE = 0.11708
Fold 3: RMSLE = 0.16053
Fold 4: RMSLE = 0.12063
Fold 5: RMSLE = 0.10698

Mean RMSLE: 0.12611
Std RMSLE:  0.01824
R² Score:   0.92158


In [26]:
results.loc[len(results)] = [
    "XGBoost",
    xgb_mean,
    xgb_std
]

results = results.sort_values("Mean RMSLE").reset_index(drop=True)
results

,Model,Mean RMSLE,Std RMSLE
0,XGBoost,0.126111,0.018238
1,Gradient Boosting,0.129890,0.018164
2,Extra Trees,0.136876,0.017676
3,Ridge,0.142617,0.040268


In [27]:
from catboost import CatBoostRegressor

In [28]:
catboost = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        CatBoostRegressor(
            iterations=500,
            learning_rate=0.03,
            depth=6,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=False,
            thread_count=-1
        )
    )
])

catboost_oof, catboost_fold_scores, catboost_mean, catboost_std = evaluate_model(
    catboost,
    X,
    y,
    kf
)

Fold 1: RMSLE = 0.13043
Fold 2: RMSLE = 0.11196
Fold 3: RMSLE = 0.15557
Fold 4: RMSLE = 0.12341
Fold 5: RMSLE = 0.10446

Mean RMSLE: 0.12516
Std RMSLE:  0.01765
R² Score:   0.92559


In [29]:
results.loc[len(results)] = [
    "CatBoost",
    catboost_mean,
    catboost_std
]

results = results.sort_values("Mean RMSLE").reset_index(drop=True)
results

,Model,Mean RMSLE,Std RMSLE
0,CatBoost,0.125165,0.017653
1,XGBoost,0.126111,0.018238
2,Gradient Boosting,0.129890,0.018164
3,Extra Trees,0.136876,0.017676
4,Ridge,0.142617,0.040268


## Phase 3 — Model Selection

In [30]:
candidate_models = {
    "CatBoost": catboost,
    "XGBoost": xgb
}

results[["Model", "Mean RMSLE", "Std RMSLE"]]

,Model,Mean RMSLE,Std RMSLE
0,CatBoost,0.125165,0.017653
1,XGBoost,0.126111,0.018238
2,Gradient Boosting,0.129890,0.018164
3,Extra Trees,0.136876,0.017676
4,Ridge,0.142617,0.040268


## Phase 4 — Hyperparameter Optimization

In [31]:
catboost_params = {
    "depth": [4, 5, 6, 7],
    "learning_rate": [0.02, 0.03, 0.05],
    "l2_leaf_reg": [3, 5, 10],
    "iterations": [500, 800, 1200]
}

In [32]:
from sklearn.model_selection import RandomizedSearchCV

catboost_search = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        CatBoostRegressor(
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=False,
            thread_count=-1
        )
    )
])

search = RandomizedSearchCV(
    estimator=catboost_search,
    param_distributions={
        f"model__{param}": values
        for param, values in catboost_params.items()
    },
    n_iter=15,
    scoring="neg_root_mean_squared_error",
    cv=kf,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1
)

search.fit(X, y)

print("Best parameters:")
print(search.best_params_)
print(f"\nBest CV score: {-search.best_score_:.5f}")

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best parameters:
{'model__learning_rate': 0.05, 'model__l2_leaf_reg': 5, 'model__iterations': 800, 'model__depth': 7}

Best CV score: 0.12300


In [33]:
tuned_catboost = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        CatBoostRegressor(
            iterations=800,
            learning_rate=0.05,
            depth=7,
            l2_leaf_reg=5,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=False,
            thread_count=-1
        )
    )
])

tuned_catboost_oof, tuned_catboost_fold_scores, tuned_catboost_mean, tuned_catboost_std = evaluate_model(
    tuned_catboost,
    X,
    y,
    kf
)

Fold 1: RMSLE = 0.12660
Fold 2: RMSLE = 0.10852
Fold 3: RMSLE = 0.15290
Fold 4: RMSLE = 0.12210
Fold 5: RMSLE = 0.10488

Mean RMSLE: 0.12300
Std RMSLE:  0.01700
R² Score:   0.92467


In [34]:
results.loc[len(results)] = [
    "CatBoost Tuned",
    tuned_catboost_mean,
    tuned_catboost_std
]

results = results.sort_values("Mean RMSLE").reset_index(drop=True)
results

,Model,Mean RMSLE,Std RMSLE
0,CatBoost Tuned,0.122999,0.017003
1,CatBoost,0.125165,0.017653
2,XGBoost,0.126111,0.018238
3,Gradient Boosting,0.129890,0.018164
4,Extra Trees,0.136876,0.017676
5,Ridge,0.142617,0.040268


In [35]:
xgb_params = {
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.02, 0.03, 0.05],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "reg_alpha": [0, 0.01, 0.1],
    "reg_lambda": [1, 5, 10],
    "n_estimators": [500, 800, 1200]
}

In [36]:
xgb_search_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

xgb_search = RandomizedSearchCV(
    estimator=xgb_search_model,
    param_distributions={
        f"model__{param}": values
        for param, values in xgb_params.items()
    },
    n_iter=15,
    scoring="neg_root_mean_squared_error",
    cv=kf,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1
)

xgb_search.fit(X, y)

print("Best parameters:")
print(xgb_search.best_params_)
print(f"\nBest CV score: {-xgb_search.best_score_:.5f}")

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best parameters:
{'model__subsample': 0.8, 'model__reg_lambda': 10, 'model__reg_alpha': 0.1, 'model__n_estimators': 800, 'model__min_child_weight': 1, 'model__max_depth': 4, 'model__learning_rate': 0.03, 'model__colsample_bytree': 0.8}

Best CV score: 0.12543


In [37]:
tuned_xgb = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        XGBRegressor(
            n_estimators=800,
            learning_rate=0.03,
            max_depth=4,
            min_child_weight=1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=10,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

tuned_xgb_oof, tuned_xgb_fold_scores, tuned_xgb_mean, tuned_xgb_std = evaluate_model(
    tuned_xgb,
    X,
    y,
    kf
)

Fold 1: RMSLE = 0.12992
Fold 2: RMSLE = 0.11717
Fold 3: RMSLE = 0.15363
Fold 4: RMSLE = 0.12034
Fold 5: RMSLE = 0.10610

Mean RMSLE: 0.12543
Std RMSLE:  0.01602
R² Score:   0.92867


In [38]:
results.loc[len(results)] = [
    "XGBoost Tuned",
    tuned_xgb_mean,
    tuned_xgb_std
]

results = results.sort_values("Mean RMSLE").reset_index(drop=True)
results

,Model,Mean RMSLE,Std RMSLE
0,CatBoost Tuned,0.122999,0.017003
1,CatBoost,0.125165,0.017653
2,XGBoost Tuned,0.125430,0.016020
3,XGBoost,0.126111,0.018238
4,Gradient Boosting,0.129890,0.018164
5,Extra Trees,0.136876,0.017676
6,Ridge,0.142617,0.040268


## Phase 5 — OOF Ensembling

In [39]:
blend_oof = (
    0.5 * tuned_catboost_oof +
    0.5 * tuned_xgb_oof
)

blend_pred = np.maximum(np.expm1(blend_oof), 0)
actual = np.maximum(np.expm1(y), 0)

blend_rmsle = np.sqrt(
    mean_squared_log_error(actual, blend_pred)
)

print(f"Equal-weight blend RMSLE: {blend_rmsle:.5f}")

Equal-weight blend RMSLE: 0.12370


In [40]:
weights = np.linspace(0, 1, 101)

blend_scores = []

for cat_weight in weights:
    xgb_weight = 1 - cat_weight

    blended_oof = (
        cat_weight * tuned_catboost_oof +
        xgb_weight * tuned_xgb_oof
    )

    blended_pred = np.maximum(np.expm1(blended_oof), 0)

    score = np.sqrt(
        mean_squared_log_error(actual, blended_pred)
    )

    blend_scores.append(score)

best_idx = np.argmin(blend_scores)
best_cat_weight = weights[best_idx]
best_xgb_weight = 1 - best_cat_weight
best_blend_rmsle = blend_scores[best_idx]

print(f"Best CatBoost weight: {best_cat_weight:.2f}")
print(f"Best XGBoost weight:  {best_xgb_weight:.2f}")
print(f"Best blend RMSLE:    {best_blend_rmsle:.5f}")

Best CatBoost weight: 0.68
Best XGBoost weight:  0.32
Best blend RMSLE:    0.12349


## Phase 6 — Error Analysis & Final Selection

In [ ]:
log_actual = y.values
log_pred = tuned_catboost_oof

residuals = log_actual - log_pred

error_analysis = pd.DataFrame({
    "Actual": log_actual,
    "Predicted": log_pred,
    "Residual": residuals,
    "Absolute_Error": np.abs(residuals)
})

print(f"Mean residual: {residuals.mean():.5f}")
print(f"Mean absolute error: {np.abs(residuals).mean():.5f}")
print(f"Maximum absolute error: {np.abs(residuals).max():.5f}")

Mean residual: -0.00196
Mean absolute error: 0.07985
Maximum absolute error: 1.25754


In [42]:
error_analysis.sort_values(
    "Absolute_Error",
    ascending=False
).head(10)

,Actual,Predicted,Residual,Absolute_Error
1298,11.982935,13.240475,-1.257540,1.257540
523,12.126764,13.365666,-1.238902,1.238902
632,11.320566,12.073606,-0.753040,0.753040
30,10.596660,11.339407,-0.742747,0.742747
495,10.460271,11.167214,-0.706943,0.706943
916,10.471978,11.148179,-0.676201,0.676201
462,11.041064,11.716432,-0.675368,0.675368
1324,11.898195,12.552038,-0.653844,0.653844
968,10.542733,11.116487,-0.573754,0.573754
410,11.002117,11.521409,-0.519293,0.519293


## Phase 7 — Final Training & Submission

In [43]:
final_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    (
        "model",
        CatBoostRegressor(
            iterations=800,
            learning_rate=0.05,
            depth=7,
            l2_leaf_reg=5,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=False,
            thread_count=-1
        )
    )
])

final_model.fit(X, y)

print("Final model trained on the full training dataset.")

Final model trained on the full training dataset.


In [44]:
test_pred_log = final_model.predict(X_test)

test_predictions = np.maximum(
    np.expm1(test_pred_log),
    0
)

print(f"Predictions generated: {len(test_predictions)}")
print(f"Minimum prediction: {test_predictions.min():,.2f}")
print(f"Maximum prediction: {test_predictions.max():,.2f}")
print(f"Mean prediction: {test_predictions.mean():,.2f}")

Predictions generated: 1459
Minimum prediction: 40,845.82
Maximum prediction: 491,326.87
Mean prediction: 175,881.48


In [45]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": test_predictions
})

print("Submission shape:", submission.shape)
print("\nMissing values:")
print(submission.isnull().sum())

print("\nNegative predictions:", (submission["SalePrice"] < 0).sum())
print("\nDuplicate IDs:", submission["Id"].duplicated().sum())

submission.head()

Submission shape: (1459, 2)

Missing values:
Id           0
SalePrice    0
dtype: int64

Negative predictions: 0

Duplicate IDs: 0


,Id,SalePrice
0,1461,124883.517948
1,1462,161942.816193
2,1463,176993.211742
3,1464,193646.422920
4,1465,184582.740177


In [46]:
assert submission.shape == (1459, 2)
assert submission["Id"].equals(test["Id"])
assert submission["SalePrice"].notna().all()
assert (submission["SalePrice"] >= 0).all()
assert submission["Id"].is_unique

print("Submission validation passed.")

Submission validation passed.


In [48]:
submission.to_csv("../submissions/submission.csv", index=False)

print("submission.csv saved successfully.")

submission.csv saved successfully.


## Final Result

The tuned CatBoost model was selected as the final model based on fixed 5-fold cross-validation.

- CV RMSLE: 0.12300
- CV standard deviation: 0.01700
- Final model: CatBoost
- Ensemble: Rejected because the optimized blend did not improve CV RMSLE
- Submission: `submission.csv`

In [49]:
results = results.sort_values(
    "Mean RMSLE",
    ascending=True
).reset_index(drop=True)

results

,Model,Mean RMSLE,Std RMSLE
0,CatBoost Tuned,0.122999,0.017003
1,CatBoost,0.125165,0.017653
2,XGBoost Tuned,0.125430,0.016020
3,XGBoost,0.126111,0.018238
4,Gradient Boosting,0.129890,0.018164
5,Extra Trees,0.136876,0.017676
6,Ridge,0.142617,0.040268
